# Drug Response Prediction with JAX

## Course Recap Project — Getting Started with JAX

**Scenario**: A precision oncology lab has gene expression profiles for hundreds of cancer cell lines and wants to predict how each cell line responds to a candidate drug compound. The response is measured as **IC50** (half-maximal inhibitory concentration in µM) — lower IC50 means the cell line is more sensitive to the drug.

We will build a **two-layer neural network from scratch** using only JAX, training it to predict log-IC50 from a panel of 12 gene expression features (e.g., *EGFR*, *TP53*, *BRCA1*, *KRAS*, *MYC*, *PIK3CA*, *PTEN*, *BRAF*, *CDK4*, *ERBB2*, *VEGFA*, *AKT1*).

This notebook ties together every major concept from the course:

| Chapter | Topic | How we use it |
|---------|-------|---------------|
| 1 | JAX Fundamentals | Arrays, PRNG, device placement |
| 2 | JIT Compilation | Accelerate the training step |
| 3 | Automatic Differentiation | `grad` / `value_and_grad` for backprop |
| 4 | Vectorization | `vmap` for batch predictions |
| 5 | Parallelization | Conceptual discussion of `pmap` scaling |
| 6 | State Management | PyTrees for model params & optimizer state |

In [ ]:
import jax
import jax.numpy as jnp
from jax import random, jit, grad, value_and_grad, vmap
import matplotlib.pyplot as plt
import time

print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")

---

## 1 — JAX Fundamentals (Chapter 1 Recap)

JAX arrays are **immutable** — you cannot modify them in-place. Instead, every operation returns a new array. JAX also replaces NumPy's global random state with an **explicit PRNG key** system, which makes randomness reproducible and compatible with JIT and parallelism.

In [ ]:
# --- Explicit PRNG keys ---
key = random.PRNGKey(42)
print(f"Master key: {key}")

# Split into independent sub-keys (never reuse a key!)
key, subkey1, subkey2 = random.split(key, 3)
print(f"Subkey 1:   {subkey1}")
print(f"Subkey 2:   {subkey2}")

# --- Immutable arrays ---
x = jnp.array([1.0, 2.0, 3.0])
# x[0] = 99.0  # This would raise an error!
x_new = x.at[0].set(99.0)  # Returns a new array
print(f"\nOriginal:  {x}")
print(f"Modified:  {x_new}")

# --- Device placement ---
print(f"\nArray device: {x.devices()}")

---

## 2 — Synthetic Biotech Dataset

We generate **fake but realistic** gene expression data for 500 cancer cell lines, each with 12 gene features measured in **log2 TPM** (Transcripts Per Million). The drug response (log-IC50) is a nonlinear function of these features, mimicking real pharmacogenomic datasets like GDSC or CCLE.

In [ ]:
# Configuration
N_SAMPLES = 500
N_FEATURES = 12
GENE_NAMES = [
    "EGFR", "TP53", "BRCA1", "KRAS", "MYC", "PIK3CA",
    "PTEN", "BRAF", "CDK4", "ERBB2", "VEGFA", "AKT1",
]

# Generate gene expression profiles (log2 TPM, typical range 2-14)
key, data_key, noise_key = random.split(key, 3)
X = random.normal(data_key, (N_SAMPLES, N_FEATURES)) * 2.0 + 8.0  # mean ~8, std ~2

# True (hidden) relationship: nonlinear combination of a few key genes
true_weights = jnp.array([0.4, -0.3, 0.0, 0.5, 0.2, -0.2, -0.4, 0.3, 0.0, 0.1, 0.0, -0.1])
linear_part = X @ true_weights
y = jnp.tanh(linear_part - linear_part.mean()) * 2.0 + 1.0  # log-IC50 in µM
y = y + random.normal(noise_key, (N_SAMPLES,)) * 0.3  # measurement noise

# Train / test split (80/20)
n_train = 400
X_train, X_test = X[:n_train], X[n_train:]
y_train, y_test = y[:n_train], y[n_train:]

print(f"Training set:  {X_train.shape[0]} cell lines × {X_train.shape[1]} genes")
print(f"Test set:      {X_test.shape[0]} cell lines × {X_test.shape[1]} genes")
print(f"log-IC50 range: [{float(y.min()):.2f}, {float(y.max()):.2f}] µM")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribution of drug response
axes[0].hist(y_train, bins=30, color="#6366f1", alpha=0.8, edgecolor="white")
axes[0].set_xlabel("log-IC50 (µM)")
axes[0].set_ylabel("Count")
axes[0].set_title("Drug Response Distribution")

# Heatmap of top gene correlations with response
correlations = jnp.array([jnp.corrcoef(X_train[:, i], y_train)[0, 1] for i in range(N_FEATURES)])
colors = ["#ef4444" if c < 0 else "#6366f1" for c in correlations]
axes[1].barh(GENE_NAMES, correlations, color=colors, edgecolor="white")
axes[1].set_xlabel("Correlation with log-IC50")
axes[1].set_title("Gene–Response Correlation")
axes[1].axvline(0, color="gray", linewidth=0.8)

# Scatter: top correlated gene vs response
top_gene_idx = int(jnp.argmax(jnp.abs(correlations)))
axes[2].scatter(X_train[:, top_gene_idx], y_train, alpha=0.4, s=15, color="#6366f1")
axes[2].set_xlabel(f"{GENE_NAMES[top_gene_idx]} expression (log2 TPM)")
axes[2].set_ylabel("log-IC50 (µM)")
axes[2].set_title(f"{GENE_NAMES[top_gene_idx]} vs Drug Response")

plt.tight_layout()
plt.show()

---

## 3 — Model Parameters as PyTrees (Chapter 6 Recap)

In JAX, we manage state through **pure functions** — no hidden mutable state. Model parameters are stored in a **PyTree** (nested dict), and every function explicitly receives and returns state.

Our network architecture:

$$\hat{y} = W_2 \cdot \text{relu}(W_1 \cdot x + b_1) + b_2$$

where $x \in \mathbb{R}^{12}$ (gene features), hidden layer has 32 units, and output is a scalar (predicted log-IC50).

In [ ]:
def init_params(key, n_input, n_hidden, n_output):
    """Initialize network parameters as a PyTree (nested dict).
    Uses Kaiming initialization for weights."""
    k1, k2 = random.split(key)
    params = {
        "layer1": {
            "w": random.normal(k1, (n_input, n_hidden)) * jnp.sqrt(2.0 / n_input),
            "b": jnp.zeros(n_hidden),
        },
        "layer2": {
            "w": random.normal(k2, (n_hidden, n_output)) * jnp.sqrt(2.0 / n_hidden),
            "b": jnp.zeros(n_output),
        },
    }
    return params

key, init_key = random.split(key)
params = init_params(init_key, N_FEATURES, 32, 1)

# PyTree structure inspection
print("Parameter PyTree structure:")
print(jax.tree.map(lambda x: x.shape, params))

`jax.tree.map` works on any **PyTree** — nested dicts, lists, tuples, or custom containers. JAX transformations like `grad` and `jit` understand PyTrees natively, so we can differentiate with respect to the entire parameter tree at once.

---

## 4 — Forward Pass & Vectorization (Chapter 4 Recap)

We write the forward pass for a **single sample**, then use `jax.vmap` to automatically vectorize it across a batch. This is cleaner than manually handling batch dimensions.

**Key idea**: `vmap(f)` transforms a function that operates on one example into a function that operates on a batch — with no loops and full hardware utilization.

In [ ]:
def predict_single(params, x):
    """Forward pass for a SINGLE sample x of shape (n_features,)."""
    h = x @ params["layer1"]["w"] + params["layer1"]["b"]  # linear
    h = jax.nn.relu(h)                                       # activation
    out = h @ params["layer2"]["w"] + params["layer2"]["b"]  # output
    return out.squeeze()  # scalar

# Vectorize over the first argument's batch dimension
# in_axes=(None, 0) means: don't batch params, batch x along axis 0
predict_batch = vmap(predict_single, in_axes=(None, 0))

# Test on a single sample
y_single = predict_single(params, X_test[0])
print(f"Single prediction: {float(y_single):.4f}")

# Test on the full batch
y_batch = predict_batch(params, X_test)
print(f"Batch predictions:  shape={y_batch.shape}, first={float(y_batch[0]):.4f}")
print(f"Match: {jnp.allclose(y_single, y_batch[0])}")

Notice how `predict_single` has **no batch dimension logic** — it's written for one sample. `vmap` handles the rest. The `in_axes` argument tells JAX which inputs to map over:

- `None` → shared across the batch (parameters)
- `0` → map along axis 0 (samples)

---

## 5 — Loss Function & Automatic Differentiation (Chapter 3 Recap)

We use **Mean Squared Error** as our loss:

$$\mathcal{L}(\theta) = \frac{1}{N} \sum_{i=1}^{N} \left( \hat{y}_i - y_i \right)^2$$

`jax.grad` computes $\nabla_\theta \mathcal{L}$ automatically via reverse-mode autodiff. Since `params` is a PyTree, `grad` returns a **gradient PyTree** with the same structure.

In [ ]:
def mse_loss(params, X, y):
    """Mean squared error over a batch."""
    preds = predict_batch(params, X)
    return jnp.mean((preds - y) ** 2)

# Compute loss
loss_val = mse_loss(params, X_train, y_train)
print(f"Initial loss: {float(loss_val):.4f}")

# --- jax.grad: returns gradient function ---
grad_fn = grad(mse_loss)  # differentiates w.r.t. first argument (params)
grads = grad_fn(params, X_train, y_train)

print(f"\nGradient PyTree structure (same as params):")
print(jax.tree.map(lambda x: x.shape, grads))
print(f"\nGrad norm (layer1/w): {float(jnp.linalg.norm(grads['layer1']['w'])):.4f}")

In [ ]:
# --- jax.value_and_grad: get loss AND gradients in one pass ---
loss_and_grad_fn = value_and_grad(mse_loss)
loss_val, grads = loss_and_grad_fn(params, X_train, y_train)
print(f"Loss: {float(loss_val):.4f}  (computed jointly with gradients)")

# --- Higher-order derivatives (grad of grad) ---
# Example: second derivative of a simple function
def f(x):
    return jnp.sin(x) * jnp.exp(-0.1 * x**2)

df = grad(f)        # first derivative
d2f = grad(df)      # second derivative

x0 = 1.0
print(f"\nHigher-order derivatives at x={x0}:")
print(f"  f(x)   = {float(f(x0)):.4f}")
print(f"  f'(x)  = {float(df(x0)):.4f}")
print(f"  f''(x) = {float(d2f(x0)):.4f}")

---

## 6 — JIT Compilation (Chapter 2 Recap)

When you call a regular Python function with JAX arrays, each operation dispatches one-at-a-time to the accelerator. `jax.jit` **traces** the function once (replacing concrete values with abstract shapes/dtypes), compiles it to an optimized XLA program, and caches it.

**Tracing caveat**: Python control flow that depends on array *values* (not shapes) breaks tracing. Use `jax.lax.cond` or `static_argnums` when needed.

Let's define the training step and compare JIT vs non-JIT speed.

In [ ]:
def train_step(params, X, y, lr):
    """One gradient-descent update. Returns (new_params, loss)."""
    loss_val, grads = value_and_grad(mse_loss)(params, X, y)
    # SGD update: θ ← θ - α∇L  (using jax.tree.map over the PyTree)
    new_params = jax.tree.map(lambda p, g: p - lr * g, params, grads)
    return new_params, loss_val

# JIT-compiled version
train_step_jit = jit(train_step)

# Warm up JIT (first call triggers tracing + compilation)
_ = train_step_jit(params, X_train, y_train, 1e-3)

# Benchmark: no JIT
t0 = time.perf_counter()
for _ in range(50):
    _ = train_step(params, X_train, y_train, 1e-3)
time_no_jit = time.perf_counter() - t0

# Benchmark: with JIT
t0 = time.perf_counter()
for _ in range(50):
    _ = train_step_jit(params, X_train, y_train, 1e-3)
time_jit = time.perf_counter() - t0

print(f"50 steps WITHOUT jit: {time_no_jit:.3f}s")
print(f"50 steps WITH    jit: {time_jit:.3f}s")
print(f"Speedup: {time_no_jit / time_jit:.1f}×")

---

## 7 — Training Loop with State Management (Chapter 6 Recap)

In JAX, state is managed **explicitly** — every function receives state as input and returns updated state as output. No global mutable variables.

The update rule is **SGD with momentum**:

$$v_t = \beta \, v_{t-1} + \nabla_\theta \mathcal{L}$$
$$\theta_{t+1} = \theta_t - \alpha \, v_t$$

We pack both `params` and `velocity` into a single state PyTree.

In [ ]:
def init_optimizer_state(params):
    """Initialize momentum velocity to zeros (same structure as params)."""
    velocity = jax.tree.map(jnp.zeros_like, params)
    return {"params": params, "velocity": velocity}

@jit
def train_step_momentum(state, X, y, lr, beta):
    """One SGD-with-momentum update. Pure function: state in → state out."""
    params = state["params"]
    velocity = state["velocity"]

    loss_val, grads = value_and_grad(mse_loss)(params, X, y)

    # Update velocity and params
    new_velocity = jax.tree.map(lambda v, g: beta * v + g, velocity, grads)
    new_params = jax.tree.map(lambda p, v: p - lr * v, params, new_velocity)

    new_state = {"params": new_params, "velocity": new_velocity}
    return new_state, loss_val

In [ ]:
# Re-initialize
key, init_key = random.split(key)
params = init_params(init_key, N_FEATURES, 32, 1)
state = init_optimizer_state(params)

# Hyperparameters
LEARNING_RATE = 3e-3
MOMENTUM = 0.9
N_EPOCHS = 500

# Training loop
train_losses = []
test_losses = []

for epoch in range(N_EPOCHS):
    state, loss_val = train_step_momentum(
        state, X_train, y_train, LEARNING_RATE, MOMENTUM
    )
    train_losses.append(float(loss_val))

    if epoch % 50 == 0 or epoch == N_EPOCHS - 1:
        test_loss = float(mse_loss(state["params"], X_test, y_test))
        test_losses.append((epoch, test_loss))
        print(f"Epoch {epoch:4d} | Train MSE: {float(loss_val):.4f} | Test MSE: {test_loss:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses, color="#6366f1", alpha=0.8, label="Train MSE")
test_epochs, test_vals = zip(*test_losses)
ax.scatter(test_epochs, test_vals, color="#ef4444", zorder=5, s=40, label="Test MSE")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("Training Progress")
ax.legend()
ax.set_yscale("log")
plt.tight_layout()
plt.show()

---

## 8 — Evaluation

In [ ]:
# Final predictions
trained_params = state["params"]
y_pred_train = predict_batch(trained_params, X_train)
y_pred_test = predict_batch(trained_params, X_test)

# Metrics
train_mse = float(jnp.mean((y_pred_train - y_train) ** 2))
test_mse = float(jnp.mean((y_pred_test - y_test) ** 2))

# R² score
ss_res = float(jnp.sum((y_pred_test - y_test) ** 2))
ss_tot = float(jnp.sum((y_test - jnp.mean(y_test)) ** 2))
r2 = 1 - ss_res / ss_tot

print(f"Train MSE: {train_mse:.4f}")
print(f"Test  MSE: {test_mse:.4f}")
print(f"Test  R²:  {r2:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Predicted vs Actual
axes[0].scatter(y_test, y_pred_test, alpha=0.5, s=20, color="#6366f1")
lims = [min(float(y_test.min()), float(y_pred_test.min())) - 0.2,
        max(float(y_test.max()), float(y_pred_test.max())) + 0.2]
axes[0].plot(lims, lims, "--", color="#ef4444", linewidth=1.5, label="Perfect")
axes[0].set_xlabel("Actual log-IC50 (µM)")
axes[0].set_ylabel("Predicted log-IC50 (µM)")
axes[0].set_title(f"Predicted vs Actual (R² = {r2:.3f})")
axes[0].legend()
axes[0].set_aspect("equal")

# Residuals
residuals = y_pred_test - y_test
axes[1].hist(residuals, bins=25, color="#6366f1", alpha=0.8, edgecolor="white")
axes[1].axvline(0, color="#ef4444", linewidth=1.5, linestyle="--")
axes[1].set_xlabel("Residual (predicted − actual)")
axes[1].set_ylabel("Count")
axes[1].set_title("Residual Distribution")

plt.tight_layout()
plt.show()

---

## 9 — Scaling with pmap (Chapter 5 Recap)

`jax.pmap` (parallel map) follows the **SPMD** (Single Program, Multiple Data) model: the same function runs simultaneously on multiple devices, each processing a shard of the data.

In a multi-GPU setting, we would distribute our training like this:

```python
# Shard data across devices
n_devices = jax.local_device_count()
X_sharded = X_train.reshape(n_devices, -1, N_FEATURES)
y_sharded = y_train.reshape(n_devices, -1)

# pmap the training step
@jax.pmap
def parallel_train_step(params, X, y, lr):
    loss, grads = value_and_grad(mse_loss)(params, X, y)
    # Average gradients across devices
    grads = jax.lax.pmean(grads, axis_name="devices")
    new_params = jax.tree.map(lambda p, g: p - lr * g, params, grads)
    return new_params, loss
```

Key concepts:
- **`in_axes`**: controls which argument axis maps to which device
- **`jax.lax.pmean` / `psum`**: collective operations to aggregate across devices
- **Replicated params**: each device holds a copy; gradients are averaged with `pmean`

Since we are running on a single device here, we won't execute `pmap`, but the pattern is identical to `vmap` — just across devices instead of batch elements.

---

## 10 — Composing Transformations

The power of JAX is that `jit`, `grad`, `vmap`, and `pmap` are **composable** — they can wrap each other freely. Our training pipeline already combines several:

```
jit(                            ← Chapter 2: compile everything
  value_and_grad(               ← Chapter 3: autodiff
    mse_loss(                   ← loss function
      vmap(predict_single)      ← Chapter 4: vectorize predictions
    )
  )
)
```

Let's verify this works and profile the composed pipeline.

In [ ]:
# Compose: jit + vmap + grad
@jit
def composed_pipeline(params, X, y):
    """Full forward + backward pass, JIT-compiled."""
    def loss_fn(p):
        preds = vmap(predict_single, in_axes=(None, 0))(p, X)
        return jnp.mean((preds - y) ** 2)
    return value_and_grad(loss_fn)(params)

# Warm up
loss_val, grads = composed_pipeline(trained_params, X_test, y_test)

# Benchmark
t0 = time.perf_counter()
for _ in range(200):
    loss_val, grads = composed_pipeline(trained_params, X_test, y_test)
time_composed = time.perf_counter() - t0

print(f"200 forward+backward passes: {time_composed:.3f}s ({time_composed/200*1000:.2f}ms each)")
print(f"Loss: {float(loss_val):.4f}")

---

## Summary

**What we built**: A two-layer neural network trained from scratch in pure JAX to predict cancer drug sensitivity from gene expression profiles.

**Course concepts applied**:

| Chapter | Concept | Where we used it |
|---------|---------|------------------|
| 1 — Fundamentals | Immutable arrays, explicit PRNG keys | Data generation, weight initialization |
| 2 — JIT | `jax.jit` for compilation, tracing | `train_step_momentum` compiled for speedup |
| 3 — Autodiff | `grad`, `value_and_grad`, higher-order derivatives | Computing gradients of MSE loss w.r.t. all params |
| 4 — vmap | `vmap(predict_single)` with `in_axes` | Batch predictions without rewriting the forward pass |
| 5 — pmap | SPMD pattern, `pmean` for gradient sync | Conceptual multi-device training pattern |
| 6 — State | PyTrees for params + velocity, pure functions | `init_optimizer_state`, `train_step_momentum` |

**Key takeaways**:
- JAX transformations (`jit`, `grad`, `vmap`, `pmap`) are **composable** — write simple code, let JAX optimize it
- **Functional purity** (no side effects, explicit state) is not just a constraint — it enables all of JAX's transformations
- **PyTrees** are the universal data structure for organizing model state
- `vmap` eliminates manual batching; `jit` eliminates Python overhead; `grad` eliminates manual calculus